<a href="https://colab.research.google.com/github/dylanhogg/jupyter-experiments/blob/fine-tuning/notebooks/finetuning/llama-factory/examples/Finetune_Llama3_with_LLaMA_Factory_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Finetune Llama-3 with LLaMA Factory v2 (llm-address)

Please use a **free** Tesla T4 Colab GPU to run this!

Project homepage: https://github.com/hiyouga/LLaMA-Factory

Original source: https://colab.research.google.com/drive/1eRTPn37ltBbYsISy9Aw2NuI2Aq5CQrD9?usp=sharing

This adaption (Sep 2025): https://github.com/dylanhogg/jupyter-experiments/blob/fine-tuning/notebooks/finetuning/llama-factory/examples/Finetune_Llama3_with_LLaMA_Factory_v2.ipynb

Training data: https://huggingface.co/datasets/dylanhogg/gnaf-2022-structured-training-100000-v0-instruct

## Imports

In [ ]:
from google.colab import files

## HF Auth

HF auth can be required for gated models you need to be granted access to.

In [ ]:
!hf auth login

## Variables

In [ ]:
# https://huggingface.co/unsloth/models?sort=downloads&search=llama
# model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
# model_name_or_path="unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
# model_name_or_path="unsloth/Llama-3.2-1B-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model

# https://huggingface.co/meta-llama/models?sort=downloads&search=llama
# model_name_or_path="meta-llama/Llama-3.2-3B-Instruct"
model_name_or_path="meta-llama/Llama-3.2-1B-Instruct"

In [ ]:
# Local backup of original dataset_info.json
!cp data/dataset_info.json data/dataset_info_original.json

In [ ]:
# Create new custom dataset_info.json
# Ref: https://github.com/hiyouga/LLaMA-Factory/blob/main/data/dataset_info.json
dataset_info = {
    "gnaf-2022-structured-training-100000-v0-instruct-train": {
        "hf_hub_url": "dylanhogg/gnaf-2022-structured-training-100000-v0-instruct",
        "split": "train",
        "columns": {
          "prompt": "instruction",
          "query": "input",
          "response": "output"
        },
    }
}
json.dump(dataset_info, open("data/dataset_info.json", "w", encoding="utf-8"), indent=2)

In [ ]:
# dataset="identity,alpaca_en_demo"
# dataset="identity"
dataset="gnaf-2022-structured-training-100000-v0-instruct-train"

# num_train_epochs=3.0
num_train_epochs=1.0

In [ ]:
!cat dataset_info.json

## Install Dependencies

In [ ]:
%cd /content/
%rm -rf LLaMA-Factory
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
%ls
!pip install -e .[torch,bitsandbytes]

### Check GPU environment

In [ ]:
import torch
try:
  assert torch.cuda.is_available() is True
except AssertionError:
  print("Please set up a GPU before using LLaMA Factory: https://medium.com/mlearning-ai/training-yolov4-on-google-colab-316f8fff99c6")

## Test Dataset

In [ ]:
from datasets import load_dataset

print(f"{dataset=}")
ds = load_dataset(dataset)
print(ds)

In [ ]:
# import json

# %cd /content/LLaMA-Factory/

# NAME = "Dylan"
# AUTHOR = "The Universe"

# with open("data/identity.json", "r", encoding="utf-8") as f:
#   dataset = json.load(f)

# for sample in dataset:
#   sample["output"] = sample["output"].replace("{{"+ "name" + "}}", NAME).replace("{{"+ "author" + "}}", AUTHOR)

# with open("data/identity.json", "w", encoding="utf-8") as f:
#   json.dump(dataset, f, indent=2, ensure_ascii=False)

## Fine-tune model via Command Line

It takes ~30min for training.

In [ ]:
import json

args = dict(
  stage="sft",                                               # do supervised fine-tuning
  do_train=True,
  model_name_or_path=model_name_or_path,
  dataset=dataset,
  # split="train",
  template="llama3",                                         # use llama3 prompt template
  finetuning_type="lora",                                    # use LoRA adapters to save memory
  lora_target="all",                                         # attach LoRA adapters to all linear layers
  output_dir="llama3_lora",                                  # the path to save LoRA adapters
  per_device_train_batch_size=2,                             # the micro batch size
  gradient_accumulation_steps=4,                             # the gradient accumulation steps
  lr_scheduler_type="cosine",                                # use cosine learning rate scheduler
  logging_steps=5,                                           # log every 5 steps
  warmup_ratio=0.1,                                          # use warmup scheduler
  save_steps=1000,                                           # save checkpoint every 1000 steps
  learning_rate=5e-5,                                        # the learning rate
  num_train_epochs=num_train_epochs,
  max_samples=500,                                           # use 500 examples in each dataset
  max_grad_norm=1.0,                                         # clip gradient norm to 1.0
  loraplus_lr_ratio=16.0,                                    # use LoRA+ algorithm with lambda=16.0
  fp16=True,                                                 # use float16 mixed precision training
  report_to="none",                                          # disable wandb logging
)

json.dump(args, open("train_llama3.json", "w", encoding="utf-8"), indent=2)

%cd /content/LLaMA-Factory/

!llamafactory-cli train train_llama3.json

In [ ]:
!ls /content/LLaMA-Factory/llama3_lora

In [ ]:
!zip -r llama_lora_results.zip /content/LLaMA-Factory/llama3_lora

In [ ]:
!ls -lha /content/LLaMA-Factory/llama_lora_results.zip

In [ ]:
from google.colab import files
files.download("/content/LLaMA-Factory/llama_lora_results.zip")

## Infer the fine-tuned model

In [25]:
from llamafactory.chat import ChatModel
from llamafactory.extras.misc import torch_gc

%cd /content/LLaMA-Factory/

args = dict(
  # model_name_or_path="unsloth/llama-3-8b-Instruct-bnb-4bit", # use bnb-4bit-quantized Llama-3-8B-Instruct model
  model_name_or_path=model_name_or_path,
  adapter_name_or_path="llama3_lora",                        # load the saved LoRA adapters
  template="llama3",                                         # same to the one in training
  finetuning_type="lora",                                    # same to the one in training
)
chat_model = ChatModel(args)

messages = []
print("Welcome to the CLI application, use `clear` to remove the history, use `exit` to exit the application.")
while True:
  query = input("\nUser: ")
  if query.strip() == "exit":
    break
  if query.strip() == "clear":
    messages = []
    torch_gc()
    print("History has been removed.")
    continue

  messages.append({"role": "user", "content": query})
  print("Assistant: ", end="", flush=True)

  response = ""
  for new_text in chat_model.stream_chat(messages):
    print(new_text, end="", flush=True)
    response += new_text
  print()
  messages.append({"role": "assistant", "content": response})

torch_gc()

KeyboardInterrupt: Interrupted by user

## Merge the LoRA adapter and optionally upload model

NOTE: the Colab free version has merely 12GB RAM, where merging LoRA of a 8B model needs at least 18GB RAM, thus you **cannot** perform it in the free version.

In [ ]:
import json

args = dict(
  # model_name_or_path="meta-llama/Meta-Llama-3-8B-Instruct", # use official non-quantized Llama-3-8B-Instruct model
  model_name_or_path=model_name_or_path,
  adapter_name_or_path="llama3_lora",                       # load the saved LoRA adapters
  template="llama3",                                        # same to the one in training
  finetuning_type="lora",                                   # same to the one in training
  export_dir="llama3_lora_merged",                          # the path to save the merged model
  export_size=2,                                            # the file shard size (in GB) of the merged model
  export_device="cpu",                                      # the device used in export, can be chosen from `cpu` and `auto`
  # export_hub_model_id="your_id/your_model",               # the Hugging Face hub ID to upload model
)

json.dump(args, open("merge_llama3.json", "w", encoding="utf-8"), indent=2)

%cd /content/LLaMA-Factory/

!llamafactory-cli export merge_llama3.json

In [ ]:
!ls /content/LLaMA-Factory/llama3_lora_merged

In [ ]:
# NOTE: can be slow since it's merged in the large base model
!zip -r llama3_lora_merged.zip /content/LLaMA-Factory/llama3_lora_merged

In [ ]:
!ls -lha /content/LLaMA-Factory/llama3_lora_merged.zip

In [ ]:
files.download("/content/LLaMA-Factory/llama3_lora_merged.zip")